In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test  = pd.read_parquet("../data/processed/X_test.parquet")


y_train_raw = pd.read_csv("../data/processed/y_train_risk.csv", header=None)
y_test_raw  = pd.read_csv("../data/processed/y_test_risk.csv", header=None)

def parse_y(raw, X_index):
    
    if raw.shape[1] == 2:
        raw.columns = ["idx", "y"]
        raw = raw.set_index("idx")
        return raw.loc[X_index, "y"].astype(int).values
    else:
        
        return raw.iloc[:, 0].astype(int).values

y_train_risk = parse_y(y_train_raw, X_train.index)
y_test_risk  = parse_y(y_test_raw, X_test.index)

print("After fix:")
print("X_train:", X_train.shape, "y_train:", len(y_train_risk))
print("X_test :", X_test.shape,  "y_test :", len(y_test_risk))


y_train_fatigue = pd.read_csv("../data/processed/y_train_fatigue.csv", index_col=0).squeeze("columns").astype(float)
y_test_fatigue  = pd.read_csv("../data/processed/y_test_fatigue.csv", index_col=0).squeeze("columns").astype(float)

X_train.shape, X_test.shape



After fix:
X_train: (4000, 12) y_train: 4000
X_test : (1000, 12) y_test : 1000


((4000, 12), (1000, 12))

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train_risk len:", len(y_train_risk))
print("y_test_risk  len:", len(y_test_risk))
print("NaNs in X_train:", X_train.isna().sum().sum())
print("NaNs in X_test :", X_test.isna().sum().sum())

clf_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500))
])

clf_base.fit(X_train, y_train_risk)
pred = clf_base.predict(X_test)

print("Accuracy:", accuracy_score(y_test_risk, pred))
print(confusion_matrix(y_test_risk, pred))
print(classification_report(y_test_risk, pred))


X_train: (4000, 12)
X_test : (1000, 12)
y_train_risk len: 4000
y_test_risk  len: 1000
NaNs in X_train: 8
NaNs in X_test : 2
Accuracy: 0.85
[[343  55   0]
 [ 44 257  21]
 [  0  30 250]]
              precision    recall  f1-score   support

           0       0.89      0.86      0.87       398
           1       0.75      0.80      0.77       322
           2       0.92      0.89      0.91       280

    accuracy                           0.85      1000
   macro avg       0.85      0.85      0.85      1000
weighted avg       0.85      0.85      0.85      1000



In [3]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train_risk len:", len(y_train_risk))
print("y_test_risk  len:", len(y_test_risk))

X_train: (4000, 12)
X_test : (1000, 12)
y_train_risk len: 4000
y_test_risk  len: 1000


In [4]:
y_train_raw = pd.read_csv("../data/processed/y_train_risk.csv", header=None)
print("y_train_risk.csv shape:", y_train_raw.shape)
y_train_raw.head(10)

y_train_risk.csv shape: (4001, 2)


,0,1
0,NaN,y_risk
1,4120.0,2
2,4121.0,2
3,4122.0,2
4,4123.0,2
5,4124.0,2
6,4125.0,2
7,4126.0,2
8,4127.0,2
9,4128.0,2


In [5]:
y_test_raw = pd.read_csv("../data/processed/y_test_risk.csv", header=None)
print("y_test_risk.csv shape:", y_test_raw.shape)
y_test_raw.head(10)

y_test_risk.csv shape: (1001, 2)


,0,1
0,NaN,y_risk
1,1798.0,0
2,1799.0,0
3,1800.0,0
4,1801.0,0
5,1802.0,0
6,1803.0,1
7,1804.0,1
8,1805.0,2
9,1806.0,2


In [6]:
from sklearn.metrics import roc_auc_score

proba = clf_base.predict_proba(X_test)
auc_ovr = roc_auc_score(y_test_risk, proba, multi_class="ovr")
auc_ovo = roc_auc_score(y_test_risk, proba, multi_class="ovo")

print("ROC-AUC (OvR):", auc_ovr)
print("ROC-AUC (OvO):", auc_ovo)

ROC-AUC (OvR): 0.9603538288538411
ROC-AUC (OvO): 0.9611488658168096


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math

reg_base = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
reg_base.fit(X_train, y_train_fatigue)
pred_f = reg_base.predict(X_test)

rmse = math.sqrt(mean_squared_error(y_test_fatigue, pred_f))
mae = mean_absolute_error(y_test_fatigue, pred_f)
r2 = r2_score(y_test_fatigue, pred_f)

print("Fatigue RMSE:", rmse)
print("Fatigue MAE :", mae)
print("Fatigue R²  :", r2)

Fatigue RMSE: 4.992676781394838
Fatigue MAE : 4.0019232117135966
Fatigue R²  : 0.7658226875253561
